# TP1

# **Préparation des données**

## ETAPE 1

In [1]:
# Les imports nécessaires pour le TP
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import spacy
import sklearn
import rank_bm25

## ETAPE 2

In [5]:
DATA = Path("data\MIRACL-FR-Mini-30K")

In [15]:
corpus = pd.read_json(f"{DATA}\corpus.jsonl", lines=True)
queries = pd.read_json(f"{DATA}\queries.jsonl", lines=True)


In [16]:
#corpus taille
corpus.shape

(30000, 3)

In [18]:
# NB de requettes
queries.shape[0]

300

In [20]:
corpus.head(3)


,_id,title,text
0,1000019#8,Police provinciale de l'Ontario,Les services opérationnels sur le terrain sont...
1,100013#0,Jacques Offenbach,"Jacques Offenbach ( """" ) est un compositeur et..."
2,10002#36,Chimpanzé,Pour préserver la faune et lutter contre les m...


In [21]:
queries.head(3)

,_id,text
0,fr-dev-0001,Qu'est-ce que c'est une famille patricienne?
1,fr-dev-0002,Pourquoi est-ce que Saint Patrick est célèbre?
2,fr-dev-0003,Pourquoi on a besoin de charbon?


In [24]:
corpus.columns


Index(['_id', 'title', 'text'], dtype='str')

In [25]:
queries.columns

Index(['_id', 'text'], dtype='str')

## ETAPE 3

In [27]:
corpus.isna().sum()

_id      0
title    0
text     0
dtype: int64

Pas de valeur manquantes à remplacer

In [ ]:
# Strip leading and trailing whitespace from the 'title' and 'text' columns
corpus['title'] = list(corpus['title'].str.strip())
corpus['text'] = list(corpus['text'].str.strip())



In [39]:
corpus['texte_complet'] = corpus['title']+" "+corpus['text']

In [40]:
corpus['texte_complet']

0        Police provinciale de l'Ontario Les services o...
1        Jacques Offenbach Jacques Offenbach ( "" ) est...
2        Chimpanzé Pour préserver la faune et lutter co...
3        Industrie nucléaire Pour ces derniers métaux, ...
4        Brigitte Le Brethon Professeur agrégée d'écono...
                               ...                        
29995    Booz endormi Booz endormi est un poème de Vict...
29996    45e parallèle nord En géographie, le nord est ...
29997    Khaṇḍā Le khaṇḍā (☬, en ) est le symbole relig...
29998    Thierry Lazaro En , il figure dans la liste de...
29999    Clown maléfique Le personnage du clown, stéréo...
Name: texte_complet, Length: 30000, dtype: str

## ETAPE 4

In [42]:
nlp = spacy.load( 
"fr_core_news_sm", 
disable=["parser", "ner"] 
) 

## ETAPE 5

In [43]:
def extraire_tokens(doc) -> list:

    # Tokenizing the document
    tokens = list(doc)

    # Keeping only tokens composed of letters
    tokens = [token for token in tokens if token.is_alpha]

    # Removing stop words
    tokens = [token for token in tokens if not token.is_stop]

    # Removing 1-character tokens
    tokens = [token for token in tokens if len(token) > 1]

    # Replacing tokens with their lemmas
    tokens = [token.lemma_ for token in tokens]

    # Converting lemmas to lowercase
    tokens = [token.lower() for token in tokens]

    return tokens

## ETAPE 6

In [44]:

docs = nlp.pipe(corpus["texte_complet"], batch_size=256)

corpus["tokens"] = [
    extraire_tokens(doc)
    for doc in docs
]

corpus["texte_pretraite"] = corpus["tokens"].apply(" ".join)

## ETAPE 7

In [47]:
docs = nlp.pipe(queries["text"], batch_size=256)

queries["tokens"] = [
    extraire_tokens(doc)
    for doc in docs
]

queries["texte_pretraite"] = queries["tokens"].apply(" ".join)

In [61]:
queries

,_id,text,tokens,texte_pretraite
0,fr-dev-0001,Qu'est-ce que c'est une famille patricienne?,"[famille, patricien]",famille patricien
1,fr-dev-0002,Pourquoi est-ce que Saint Patrick est célèbre?,"[saint, patrick, célèbre]",saint patrick célèbre
2,fr-dev-0003,Pourquoi on a besoin de charbon?,"[besoin, charbon]",besoin charbon
3,fr-dev-0004,Pourquoi est-ce qu'on peut construire un ecovi...,"[construire, ecovillage]",construire ecovillage
4,fr-dev-0005,Comment sont créés les astéroïdes?,"[créer, astéroïde]",créer astéroïde
...,...,...,...,...
295,fr-dev-0339,Comment mesurer l’amplitude?,"[mesurer, amplitude]",mesurer amplitude
296,fr-dev-0340,Qui est le chef du Parlement britannique?,"[chef, parlement, britannique]",chef parlement britannique
297,fr-dev-0341,Où se trouve la Principauté de Tsiakarandambo?,"[trouver, principauté, tsiakarandambo]",trouver principauté tsiakarandambo
298,fr-dev-0342,Qui a été le premier cycliste à avoir remporté...,"[être, cycliste, remporter, tour, france]",être cycliste remporter tour france


In [48]:
corpus

,_id,title,text,texte_complet,tokens,texte_pretraite
0,1000019#8,Police provinciale de l'Ontario,Les services opérationnels sur le terrain sont...,Police provinciale de l'Ontario Les services o...,"[police, provincial, ontario, service, opérati...",police provincial ontario service opérationnel...
1,100013#0,Jacques Offenbach,"Jacques Offenbach ( """" ) est un compositeur et...","Jacques Offenbach Jacques Offenbach ( """" ) est...","[jacques, offenbach, jacques, offenbach, compo...",jacques offenbach jacques offenbach compositeu...
2,10002#36,Chimpanzé,Pour préserver la faune et lutter contre les m...,Chimpanzé Pour préserver la faune et lutter co...,"[chimpanzé, préserver, faune, lutter, contre, ...",chimpanzé préserver faune lutter contre menace...
3,1000250#15,Industrie nucléaire,"Pour ces derniers métaux, la consommation de l...","Industrie nucléaire Pour ces derniers métaux, ...","[industrie, nucléaire, dernier, métal, consomm...",industrie nucléaire dernier métal consommation...
4,10004#1,Brigitte Le Brethon,Professeur agrégée d'économie et gestion dans ...,Brigitte Le Brethon Professeur agrégée d'écono...,"[brigitte, brethon, professeur, agréger, écono...",brigitte brethon professeur agréger économie g...
...,...,...,...,...,...,...
29995,999799#0,Booz endormi,Booz endormi est un poème de Victor Hugo extra...,Booz endormi Booz endormi est un poème de Vict...,"[booz, endormi, booz, endormi, poème, victor, ...",booz endormi booz endormi poème victor hugo ex...
29996,999817#0,45e parallèle nord,"En géographie, le nord est le parallèle joigna...","45e parallèle nord En géographie, le nord est ...","[parallèle, nord, géographie, nord, parallèle,...",parallèle nord géographie nord parallèle joind...
29997,999855#0,Khaṇḍā,"Le khaṇḍā (☬, en ) est le symbole religieux le...","Khaṇḍā Le khaṇḍā (☬, en ) est le symbole relig...","[khaṇḍā, khaṇḍā, symbole, religieux, important...",khaṇḍā khaṇḍā symbole religieux important sikh...
29998,9999#5,Thierry Lazaro,"En , il figure dans la liste des parlementaire...","Thierry Lazaro En , il figure dans la liste de...","[thierry, lazaro, figurer, liste, parlementair...",thierry lazaro figurer liste parlementaire par...


## ETAPE 8

In [49]:
corpus.to_pickle("corpus_pretraite.pkl")
queries.to_pickle("queries_pretraite.pkl")

# **Modèles TF-IDF**

## ETAPE 9

In [54]:
obj_vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(lowercase=False, tokenizer=str.split, preprocessor=None, token_pattern=None, min_df=2, norm='l2')

## ETAPE 10

In [55]:
X_documents = obj_vectorizer.fit_transform(corpus["texte_pretraite"])

## ETAPE 11

In [56]:
X_requetes = obj_vectorizer.transform(queries["texte_pretraite"])

## ETAPE 12

In [69]:
def rechercher_tfidf(indice_requete, k=5):
    # Calcul de similarité cosinus entre la requête et tous les documents du corpus
    scores = sklearn.metrics.pairwise.cosine_similarity(
        X_requetes[indice_requete],
        X_documents
    )[0]
    # ajout des scores de similarité à une copie du corpus
    # (on n'écrase pas la variable globale "corpus" : sinon UnboundLocalError)
    resultats = corpus.assign(score=scores)
    # Elimination des documents avec un score de similarité nul
    resultats = resultats[resultats["score"] > 0]
    # Sélection des k documents les plus pertinents
    resultats = resultats.nlargest(k, "score").copy()
    # Création d'un extrait du texte pour chaque document
    resultats["extrait"] = resultats["text"].str[:200]
    return resultats[["_id", "title", "extrait", "score"]]

# **Modèle BM25**

## ETAPE 13

In [58]:
# Construction du modèle BM25
modele_bm25 = rank_bm25.BM25Okapi( 
corpus["tokens"].tolist(), 
k1=1.2, 
b=0.75 
)

## ETAPE 14

In [68]:
def rechercher_bm25(indice_requete, k=5):
    # Récupération des tokens de la requête
    tokens_requete = queries.loc[indice_requete, "tokens"]
    # Calcul des scores BM25 pour tous les documents du corpus
    scores = modele_bm25.get_scores(tokens_requete)
    # ajout des scores de similarité à une copie du corpus
    resultats = corpus.assign(score=scores)
    # Elimination des documents avec un score de similarité nul
    resultats = resultats[resultats["score"] > 0]
    # Sélection des k documents les plus pertinents
    resultats = resultats.nlargest(k, "score").copy()
    # Création d'un extrait du texte pour chaque document
    resultats["extrait"] = resultats["text"].str[:200]
    return resultats[["_id", "title", "extrait", "score"]]

# **Applicatipn aux memes requetes**

## ETAPE 15

In [60]:
indices_requetes_test = [0, 1] 

## ETAPE 16

In [70]:
for indice in indices_requetes_test:
    # texte original + tokens après prétraitement
    print("Requête :", queries.loc[indice, "text"])
    print("Requête prétraitée :", queries.loc[indice, "texte_pretraite"])
    print("\nLes résultats de TF-IDF")
    display(rechercher_tfidf(indice))
    print("\nLes résultats de BM25")
    display(rechercher_bm25(indice))

Requête : Qu'est-ce que c'est une famille patricienne?
Requête prétraitée : famille patricien

Les résultats de TF-IDF


,_id,title,extrait,score
5430,13853489#1,Patricien (Moyen Âge et Temps modernes),"Au Moyen Âge, dans les villes italiennes, « pa...",0.696396
1845,11574018#0,Famille Patry,La famille Patry est une famille patricienne g...,0.631765
5431,13853489#2,Patricien (Moyen Âge et Temps modernes),Dans les territoires de l'ancien Saint-Empire ...,0.572838
5434,13853489#5,Patricien (Moyen Âge et Temps modernes),Nonobstant cette appartenance à une famille pa...,0.565937
25387,6873881#23,République romaine,Les premières familles romaines sont appelées ...,0.547261



Les résultats de BM25


,_id,title,extrait,score
1845,11574018#0,Famille Patry,La famille Patry est une famille patricienne g...,16.456334
14525,312255#6,Patricien,"Les ""patres"" qui remplissent les rangs du prem...",16.355745
5431,13853489#2,Patricien (Moyen Âge et Temps modernes),Dans les territoires de l'ancien Saint-Empire ...,15.767413
25387,6873881#23,République romaine,Les premières familles romaines sont appelées ...,15.507881
5430,13853489#1,Patricien (Moyen Âge et Temps modernes),"Au Moyen Âge, dans les villes italiennes, « pa...",15.189885


Requête : Pourquoi est-ce que Saint Patrick est célèbre?
Requête prétraitée : saint patrick célèbre

Les résultats de TF-IDF


,_id,title,extrait,score
6184,1413550#26,Patrick d'Irlande,"Pour les catholiques, saint Patrick est le sai...",0.530493
28225,834018#2,Itaqui,"Pour les catholiques romains, saint Patrick es...",0.437500
4510,12876#34,Guipavas,"Saint Thudon est parfois, mais cela reste très...",0.381702
26782,7767661#2,Église Saint-Valentin de Guilers,Le Saint patron de l'église est Saint Valentin...,0.377201
17272,3704726#0,Patrick Collard,"Patrick Collard, né le , est un dirigeant de g...",0.365011



Les résultats de BM25


,_id,title,extrait,score
6184,1413550#26,Patrick d'Irlande,"Pour les catholiques, saint Patrick est le sai...",17.409834
25429,691903#0,Fête de la Saint-Patrick,La fête de la Saint-Patrick (parfois appelée S...,15.441561
28225,834018#2,Itaqui,"Pour les catholiques romains, saint Patrick es...",14.565012
6183,1413550#22,Patrick d'Irlande,La biographie de Patrick est noyée sous l'hagi...,12.213731
27143,7898177#7,Sénateur de Milan,Saint Sénateur fut enterré dans cette basiliqu...,12.061037


# **Partie : Analyse des résultats**


### Question 1
Non, les deux modèles ne renvoient pas exactement les mêmes cinq documents.
- **Requête 1 (famille patricienne)** : 4 documents sur 5 sont communs. TF-IDF renvoie `13853489#5`, alors que BM25 renvoie à sa place `312255#6` (article *Patricien*).
- **Requête 2 (Saint Patrick)** : seulement 2 documents sont communs (`1413550#26` et `834018#2`).

### Question 2
Non, l'ordre diffère. Pour la requête 1, TF-IDF place *Patricien (Moyen Âge)* en tête, alors que BM25 place *Famille Patry* en premier. Pour la requête 2, le premier document est le même (*Patrick d'Irlande*), mais la suite est différente.

### Question 3
Les deux scores ne sont pas sur la même échelle :
- **TF-IDF** donne une similarité cosinus entre vecteurs normalisés (L2), donc un score compris entre 0 et 1.
- **BM25** donne une somme de poids IDF × TF saturé et normalisé par la longueur du document. Ce score n'est pas borné et dépend du nombre de termes de la requête et des statistiques du corpus.

Seul le **classement** à l'intérieur d'un même modèle est comparable, pas les valeurs brutes.

### Question 4
- **Requête 1** : les deux modèles donnent des résultats pertinents. BM25 est légèrement meilleur car il remonte l'article général *Patricien* (`312255#6`), qui définit directement la notion.
- **Requête 2** : **BM25 est nettement meilleur**. Il renvoie *Fête de la Saint-Patrick* et la biographie de *Patrick d'Irlande*. TF-IDF renvoie des documents hors sujet (*Guipavas*, *Église Saint-Valentin*, *Patrick Collard*) qui ne contiennent qu'un seul des deux mots « saint » ou « patrick ».

BM25 favorise mieux les documents qui contiennent **tous** les termes de la requête, grâce à la saturation de la fréquence des termes (k1) et à la normalisation par la longueur (b).